In [170]:
from assets import loadMelodic, select_files, corpora_parser, match_lengths, csv_export, csv_import
from music21 import *
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
import pandas as pd
import nltk
from nltk.lm import Laplace, Vocabulary, MLE
from nltk import FreqDist
from sklearn.model_selection import train_test_split, cross_val_score
from nltk.util import pad_sequence
import statistics
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
import nltk.classify.util
from sklearn.svm import SVC
from nltk.classify import NaiveBayesClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from decouple import config, AutoConfig
import os

### Opción 3. Uso de clasificadores Random Forest

In [200]:
n = 3
mode = 'Ii'
voice_selec= 3
stack = False
diatonic = False
base40=True

In [151]:
## Atención: ejecutar este código

config = AutoConfig(' ')
inp = input(f"La carpeta contenedora de la música es {config('MUSIC_FOLDER')}. Introduzca un nombre distinto o N, para conservar el nombre: ")

if inp.upper() != 'N':
    with open('.env', 'r') as f:
        lines = f.readlines()

    with open('.env', 'w') as f:
        for line in lines:
            if line.startswith('MUSIC_FOLDER='):
                f.write(f'MUSIC_FOLDER={inp}\n')
            else:
                f.write(line)
    os.environ['MUSIC_FOLDER'] = inp
else:
    os.environ['MUSIC_FOLDER'] = config('MUSIC_FOLDER')

La carpeta contenedora de la música es Music_files. Introduzca un nombre distinto o N, para conservar el nombre:  N


#### Selección de archivos para cada dataset (compositor 1, compositor 2 y anónimos)

In [152]:
morales_fnames_and_others = select_files(split_by_voices=True)
otros_fnames_and_others = select_files(split_by_voices=True, multiple_folder_selection=True)
test_fnames_and_others = select_files(split_by_voices=True)

Introduzca el nombre del directorio de obras a cargar:  Morales
El conjunto contiene obras clasificadas en 6 agrupaciones diferentes con [2, 3, 4, 5, 6, 7] voces. ¿Qué subset desea importar?  4


Los datasets disponibles son:

['Claudin', 'Consilium', 'Courtois', 'Daser', 'Escobedo', 'Festa', 'Gombert', 'Guerrero', 'Ivo', 'J. Gero', 'Jacotin', 'Jaquet of Mantua', 'Maistre Jhan', 'Palestrina', 'Peñalosa', 'Phinot', 'Ruffo', 'Verdelot', 'Willaert']


Introduzca los directorios de interés, escriba "OK" para terminar, simplemente "All" o "All-[carpeta a excluir]"":  All-Anon
El conjunto contiene obras clasificadas en 7 agrupaciones diferentes con [3, 4, 5, 6, 7, 8, 9] voces. ¿Qué subset desea importar?  4
Introduzca el nombre del directorio de obras a cargar:  Anon
El conjunto contiene obras clasificadas en 1 agrupaciones diferentes con [4] voces. ¿Qué subset desea importar?  4


#### Extraer features de compositor 1: Morales##

In [153]:
## Aclaración de uso:
#Si queremos importar todos los archivos, independientemente de su plantilla musical, y tratar todas las voces indistintamente:
#stack=True, voice_selec=All, non_splitting=True.

morales = corpora_parser(n, morales_fnames_and_others, otros_fnames_and_others, diatonic=diatonic, base40=base40, stack=stack, mode=mode, voice_selec=voice_selec)
morales_words = morales[1]
morales_ngrams = morales[0]
morales_files = morales[3]
iors_hist = Counter(morales[2])

#### Extraer features de compositor 2: Otros

In [154]:
otros = corpora_parser(n, otros_fnames_and_others, morales_fnames_and_others, diatonic=diatonic, base40=base40, stack=stack, mode=mode, voice_selec=voice_selec)
otros_words = otros[1]
otros_ngrams = otros[0]
otros_files = otros[3]
iors_hist = Counter(otros[2])

#### Exportación de los datos en CSV

In [ ]:
csv_export(morales, composer='morales')

In [ ]:
csv_export(otros, composer='otros')

#### Importación de los datos de CSV

In [197]:
morales = csv_import('Exports/morales4v_3grams_Ii_voice-3_b40.csv.gz')
morales_ngrams = morales[0]
morales_files = morales[1]
morales_words = morales[2]

In [198]:
otros = csv_import('Exports/otros4v_3grams_Ii_voice-3_b40.csv.gz')
otros_ngrams = otros[0]
otros_files = otros[1]
otros_words = otros[2]

#### Extraer features de Problemáticos: Test (obligatorio)

In [201]:
test = corpora_parser(n, test_fnames_and_others, test_fnames_and_others, diatonic=diatonic, base40=base40, stack=stack, mode=mode, voice_selec=voice_selec)
test_ngrams = test[0]
test_words = test[1]
test_files = test[3]

## Unión de listas de palabras en BoW
#### ... y fusión de dos corpus y sus etiquetas en dos variables. 

In [202]:
def removeNan(x):
    out = list()
    for i in morales_words:
        temp = list()
        for x in i:
            if isinstance(x, str):
                temp.append(x)
        out.append(temp)
    return out

morales_words = removeNan(morales_words)
otros_words = removeNan(otros_words)
test_bow = removeNan(test_words)

In [203]:
def merge_words(x):
    data = list()
    for i in x:
        y = ' '.join(i)
        data.append(y)
    return data

dataset = merge_words(morales_words)
words_otros = merge_words(otros_words)
dataset.extend(words_otros)
tags = ['Morales'] * len(morales_words)
otros_tags = ['Otros'] * len(words_otros)
tags.extend(otros_tags)

test_bow = merge_words(test_bow)

vocab = set([x for i in dataset for x in i.split()])


In [229]:
classifier = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)

# Matriz de features

## elimino , vocabulary=vocab porque ignora las palabras nuevas del test
matrix = TfidfVectorizer(lowercase=False)
X = matrix.fit_transform(dataset).toarray()

# Etiquetas de cada clase
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(tags)

# Datos y etiquetas de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Clasificador

classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

T = matrix.transform(test_bow).toarray()  # Convertir a matriz numérica
#y_pred = classifier.predict(T)  # Predicción en números
y_pred_text = label_encoder.inverse_transform(y_pred)  # Convertir a texto

print("Predicción:", y_pred_text[0])  # Salida en palabras

# Evaluación de y_pred frente a y_test
cm = confusion_matrix(y_test, y_pred)
cr = classification_report(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

Predicción: Otros


In [230]:
cm

array([[ 1, 24],
       [32,  1]])

In [231]:
round(accuracy, 2)

0.03

In [224]:
y_pred == y_test

array([ True, False, False,  True, False, False,  True, False,  True,
       False, False,  True,  True, False, False, False,  True, False,
       False, False, False, False,  True, False,  True, False,  True,
       False,  True, False,  True, False, False, False, False, False,
       False, False, False, False, False, False, False,  True,  True,
        True,  True, False, False, False, False, False,  True, False,
       False, False, False,  True])

### Experimento 2: Clasificador NB Multinomial

In [225]:
## Funciona mucho mejor como búsqueda de palabras clave. 
#No tiene en cuenta el valor relativo de las palabras dentro del corpus
#Pero funciona mejor así que normalizando y con TfidfVectorizer

classifier = RandomForestClassifier()

# Matriz de features
#matrix = CountVectorizer(lowercase=False, vocabulary=vocab)
matrix = TfidfVectorizer(lowercase=False, vocabulary=vocab)
X = matrix.fit_transform(words).toarray()

# Etiquetas de cada clase
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(tags)

# Datos y etiquetas de entrenamiento y prueba
X_train, X_test, y_train, y_test, data_train, data_test = train_test_split(X, y, dataset, test_size=0.2, random_state=42)

# Clasificador

classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

# Evaluación de y_pred frente a y_test
cm = confusion_matrix(y_test, y_pred)
cr = classification_report(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

NameError: name 'words' is not defined

In [36]:
cm

array([[47,  8],
       [20, 41]])

In [37]:
round(accuracy, 2)

0.76

In [38]:
y_pred == y_test

array([ True,  True,  True,  True,  True,  True, False, False,  True,
       False,  True,  True,  True, False,  True,  True,  True, False,
        True,  True, False,  True,  True,  True,  True,  True, False,
       False,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True, False,  True, False,  True,  True,  True,  True,
        True,  True,  True, False,  True,  True,  True, False, False,
        True, False,  True,  True,  True, False,  True,  True,  True,
        True,  True,  True,  True,  True, False,  True,  True, False,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
       False, False,  True, False,  True,  True,  True,  True,  True,
        True,  True, False,  True,  True,  True, False,  True, False,
       False, False,  True,  True,  True, False,  True, False,  True,
        True, False,  True,  True,  True,  True,  True,  True])

### Experimento 3: Tfidf con GaussianNB

In [50]:
nb = GaussianNB()

# Matriz de features
matrix = TfidfVectorizer(lowercase=False, vocabulary=vocab, norm='l2')
x = matrix.fit_transform(dataset).toarray()

print(x)
# Etiquetas de cada clase
label_encoder = LabelEncoder()
Y = label_encoder.fit_transform(tags)

# Datos y etiquetas de entrenamiento y prueba
x_train, x_test, Y_train, Y_test, data_train, data_test = train_test_split(x, Y, dataset, test_size=0.2, random_state=42)

# Clasificador

nb.fit(x_train, Y_train)

Y_pred = nb.predict(x_test)

# Evaluación de y_pred frente a y_test
cm = confusion_matrix(Y_test, Y_pred)
cr = classification_report(Y_test, Y_pred)
accuracy = accuracy_score(Y_test, Y_pred)
print(accuracy)

[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.08230588 0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]
0.13793103448275862


### Experimento 2.2

In [51]:
min_length = min([len(i.split()) for i in dataset])
min_length

# Definir una función para clasificar un documento completo dividido en subdocumentos
def classify_document(model, vectorizer, document, window_size=min_length, overlap=min_length-1):

    subdocuments = [document.split()[i:i+window_size] for i in range(0, len(document.split()), window_size - overlap)]
    predictions = []
    for subdoc in subdocuments:
        subdoc_features = vectorizer.fit_transform([' '.join(subdoc)]).toarray()
        prediction = model.predict(subdoc_features)
        predictions.append(prediction[0])
    final_prediction = np.bincount(predictions).argmax()
    
    return [tuple(predictions), f'window_size={min_length}', final_prediction]

In [52]:
# Clasificación del documento completo dividido en subdocumentos

nb = MultinomialNB(alpha=1.0)
nb.fit(X_train, y_train)

# Matriz de features
matrix = CountVectorizer(lowercase=False, vocabulary=vocab)
X = matrix.fit_transform(dataset).toarray()

rst = []
rst_full = []
for i in data_test:
    result = classify_document(nb, matrix, i)
    # Imprimir la clase predicha para el documento completo
    rst.append(result[-1])
    rst_full.append(result)

In [53]:
acc = accuracy_score(y_test, rst)
acc

0.25862068965517243

In [54]:
confusion = confusion_matrix(y_test, rst)
confusion

array([[12, 13],
       [30,  3]])

In [55]:
y_test == rst

array([ True, False,  True,  True, False, False,  True, False, False,
       False, False,  True, False, False, False, False, False, False,
       False,  True, False, False, False,  True,  True, False, False,
       False,  True,  True, False, False, False, False, False, False,
       False, False, False, False, False, False, False,  True,  True,
       False,  True, False,  True, False, False, False,  True, False,
       False, False, False, False])

### Ahora pruebas con N-gramas (que ya reflejan secuencias de tres notas)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

In [52]:
tfidf = TfidfVectorizer(min_df=2, max_df=1.0, ngram_range=(1,1))
features = tfidf.fit_transform(dataset)

In [53]:
df = pd.DataFrame(
    features.todense(),
    columns=tfidf.get_feature_names_out()
)

In [54]:
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=42)

# Clasificador
classifier = RandomForestClassifier()
classifier.fit(X_train, y_train)

# Predicciones
y_pred = classifier.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
cr = classification_report(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

In [55]:
accuracy

0.7672413793103449

In [56]:
cm

array([[50,  5],
       [22, 39]])

In [57]:
cr

'              precision    recall  f1-score   support\n\n           0       0.69      0.91      0.79        55\n           1       0.89      0.64      0.74        61\n\n    accuracy                           0.77       116\n   macro avg       0.79      0.77      0.77       116\nweighted avg       0.80      0.77      0.76       116\n'

In [58]:
def show_most_informative_features(vectorizer, clf, n=20):
    feature_names = vectorizer.get_feature_names_out()
    coefs_with_fns = sorted(zip(clf.feature_log_prob_[0], feature_names))
    top = zip(coefs_with_fns[:n], coefs_with_fns[:-(n + 1):-1])
    for (coef_1, fn_1), (coef_2, fn_2) in top:
        print("\t%.4f\t%-15s\t\t%.4f\t%-15s" % (coef_1, fn_1, coef_2, fn_2))
        
show_most_informative_features(tfidf, classifier)    

AttributeError: 'RandomForestClassifier' object has no attribute 'feature_log_prob_'